# 19.7 工具变量 / Instrumental Variables (2SLS)

**中文**：倾向匹配(19.5)有个致命前提:**所有混杂都被测到了**。但很多时候关键混杂是**测不到的**——比如研究"教育对工资的因果效应",一个人的**天赋/能力**同时影响了他"读多少书"和"赚多少钱",而能力**观测不到**。这时 PSM、回归全都失效(会把能力的功劳算给教育)。**工具变量(IV)** 是应对**未观测混杂**的经典武器:找一个只影响"教育"、但不直接影响"工资"的外部变量当"杠杆",撬出教育里"干净"的那部分变化来估因果。
**English**: Propensity matching (19.5) has a fatal prerequisite: **all confounders are measured**. But often the key confounder is **unmeasurable** — e.g. studying "education's causal effect on wages," a person's **talent/ability** affects both "how much they study" and "how much they earn," and ability is **unobserved**. Then PSM and regression all fail (they credit ability to education). **Instrumental variables (IV)** are the classic weapon against **unobserved confounding**: find an external variable that affects "education" but not "wages" directly, as a "lever" to extract the "clean" part of education's variation for causal estimation.

---

**中文**：一个合格的**工具变量 $Z$** 必须满足三个条件:
**English**: A valid **instrument $Z$** must satisfy three conditions:
- **① 相关性(relevance)**:$Z$ 确实影响处理 $X$($Z\to X$,且要够强)。可检验。
  **Relevance**: $Z$ truly affects the treatment $X$ ($Z\to X$, and strongly). Testable.
- **② 排他性(exclusion restriction)**:$Z$ **只能通过 $X$** 影响结果 $Y$,不能有别的路径($Z\to Y$ 只经 $X$)。**不可检验**——这是 IV 最脆弱的假设。
  **Exclusion restriction**: $Z$ affects $Y$ **only through $X$**, no other path ($Z\to Y$ only via $X$). **Untestable** — IV's most fragile assumption.
- **③ 独立性(independence)**:$Z$ 与未观测混杂无关(像被随机分配一样)。
  **Independence**: $Z$ is unrelated to the unobserved confounder (as if randomly assigned).

**中文**：经典工具变量的例子极巧妙:研究**参军对收入**的影响→用**越战抽签号(draft lottery)** 当工具(抽签随机、只通过"是否参军"影响收入);研究**教育对工资**→用**出生季度**(义务教育法使不同季度出生的人受教育年限略不同)或**离大学的距离**;研究**基因对疾病**→用**遗传变异(孟德尔随机化)**。
**English**: Classic instruments are ingenious: for **military service on earnings** → the **Vietnam draft lottery** (random, affects earnings only through "served or not"); for **education on wages** → **quarter of birth** (compulsory-schooling laws make schooling years differ slightly by birth quarter) or **distance to college**; for **genes on disease** → **genetic variants (Mendelian randomization)**.

**中文**：估计用 **两阶段最小二乘(2SLS)**,直觉极清晰:
**English**: Estimation uses **Two-Stage Least Squares (2SLS)**, with a clear intuition:
1. **第一阶段**:把处理 $X$ 对工具 $Z$ 回归,得到**预测值 $\hat X$**——这是 $X$ 中**由 $Z$ 驱动的那部分**,而 $Z$ 与混杂无关,所以 $\hat X$ **"干净"、不含混杂**。
   **Stage 1**: regress treatment $X$ on instrument $Z$ to get **predictions $\hat X$** — the part of $X$ **driven by $Z$**; since $Z$ is unrelated to the confounder, $\hat X$ is **"clean," confounder-free**.
2. **第二阶段**:把结果 $Y$ 对 $\hat X$ 回归,系数就是**因果效应**(因为用的是干净的 $\hat X$)。
   **Stage 2**: regress outcome $Y$ on $\hat X$; the coefficient is the **causal effect** (since it uses the clean $\hat X$).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 计量/因果必考）**
> **中文**：**IV=对付未观测混杂**(PSM/回归失效时)。工具 Z 三条件:**相关(Z→X 强, 可检验)、排他(Z 只经 X 影响 Y, 不可检验)、独立(Z 与混杂无关)**。**2SLS**:①X~Z 得 X̂(X 中被 Z 驱动的干净部分);②Y~X̂ 系数=因果效应。**弱工具问题(必考)**:Z 对 X 太弱(**第一阶段 F<10**)→ 2SLS 方差爆炸、偏向 OLS、不可靠。**估的是 LATE**(局部平均处理效应)——只对"被工具影响的那群人(complier)"有效, 不是总体 ATE。**软肋**:排他性不可检验(常被质疑)、弱工具、LATE 外推性差。判断好工具的关键就是问:"这个 Z 有没有别的路径影响 Y?"
> **English**: **IV = handle unobserved confounding** (when PSM/regression fail). Three conditions for Z: **relevance (Z→X strong, testable), exclusion (Z affects Y only via X, untestable), independence (Z unrelated to the confounder)**. **2SLS**: ① X~Z gives X̂ (the clean, Z-driven part of X); ② Y~X̂ coefficient = the causal effect. **Weak-instrument problem (a favorite)**: Z too weakly related to X (**first-stage F<10**) → 2SLS variance explodes, biases toward OLS, unreliable. **Estimates LATE** (Local Average Treatment Effect) — valid only for the "compliers" (those the instrument moves), not the population ATE. **Weaknesses**: the exclusion restriction is untestable (often disputed), weak instruments, poor external validity of LATE. The key to judging an instrument: "does this Z have any other path to Y?"


In [ ]:

# ============================================================
# 模拟:教育→工资, 能力是未观测混杂 / education→wage with unobserved ability confounder
# 中文:真实"教育回报"=0.5。能力(ability)【观测不到】, 同时抬高教育和工资(混杂)。
#      工具 Z(如"离大学距离"的反向)影响教育, 但不直接影响工资。看谁能还原 0.5。
# English: true "return to education" = 0.5. Ability is UNOBSERVED, raising both education and wage (confounding).
#      Instrument Z (e.g. reverse distance-to-college) affects education but not wage directly.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
import statsmodels.api as sm
rng=np.random.default_rng(0)
N=5000; true_return=0.5
ability = rng.normal(0,1,N)                                  # 未观测混杂:能力 / UNOBSERVED confounder
Z = rng.normal(0,1,N)                                        # 工具变量 / instrument
edu  = 0.6*Z + 0.7*ability + rng.normal(0,1,N)              # 教育=工具驱动+能力驱动 / education
wage = true_return*edu + 0.8*ability + rng.normal(0,1,N)    # 工资=教育因果+能力混杂 / wage

# ① 朴素 OLS:工资对教育回归(混杂偏差)/ naive OLS wage~edu (confounded)
naive = sm.OLS(wage, sm.add_constant(edu)).fit().params[1]
print(f"真实教育回报 / true return: {true_return}")
print(f"① 朴素 OLS / naive: {naive:.3f}  (被'能力'向上污染 → 高估)")


**中文**：朴素回归把 0.5 的真实回报高估到 ~0.78——因为"能力"同时抬高了教育和工资,这份功劳被错算给了教育(**这正是"高学历高收入,有多少是教育的功劳、有多少只是聪明人本来就赚得多"这个经典难题**)。而且**能力观测不到**,所以没法控制它——PSM 在这里彻底失效。现在用 **2SLS** 借工具 $Z$ 破局。
**English**: The naive regression overestimates the true return of 0.5 to ~0.78 — because "ability" raises both education and wages, and that credit is misattributed to education (**exactly the classic puzzle: "of the high-education-high-income link, how much is education's doing vs smart people simply earning more?"**). And since **ability is unobserved**, you cannot control for it — PSM totally fails here. Now use **2SLS** with instrument $Z$ to break through.


In [ ]:

# ============================================================
# 从零实现 2SLS / Two-Stage Least Squares from scratch
# ============================================================
# 第一阶段:教育 ~ 工具 Z → 得到"干净的"预测教育 / stage 1: edu ~ Z → clean predicted edu
stage1 = sm.OLS(edu, sm.add_constant(Z)).fit()
edu_hat = stage1.fittedvalues                               # X̂:只含被 Z 驱动的部分(不含能力)/ Z-driven part
F_stat = stage1.fvalue                                      # 第一阶段 F(工具强度)/ first-stage F

# 第二阶段:工资 ~ 预测教育 → 系数=因果效应 / stage 2: wage ~ edu_hat → causal effect
stage2 = sm.OLS(wage, sm.add_constant(edu_hat)).fit()
iv_est = stage2.params[1]

print(f"第一阶段 F 统计量 / first-stage F: {F_stat:.0f}  (>10 = 强工具, 可靠)")
print(f"② 2SLS 估计 / IV: {iv_est:.3f}  ← 还原了真值 {true_return}!")
print(f"\n对比 / comparison:  真值 {true_return}  |  朴素OLS {naive:.3f}(有偏)  |  2SLS {iv_est:.3f}(无偏)")
print("为什么有效:X̂ 只含'被工具驱动'的教育变化, 与能力无关 → 干净, 不含混杂")


**中文**：2SLS 精确还原了真实回报 0.5——它用工具 $Z$ 撬出教育中"与能力无关的干净变化",绕开了未观测混杂。但 IV 有个著名的**软肋:弱工具**。如果 $Z$ 对教育的影响太弱(第一阶段 F 太小),2SLS 就会**方差爆炸、结果极不稳定**。下面对比强/弱工具在多次重复实验里的表现。
**English**: 2SLS exactly recovers the true return 0.5 — it uses instrument $Z$ to extract education's "clean, ability-free variation," bypassing unobserved confounding. But IV has a famous **weakness: weak instruments**. If $Z$'s effect on education is too weak (first-stage F too small), 2SLS suffers **exploding variance and wildly unstable results**. Below we compare strong vs weak instruments across repeated experiments.


In [ ]:

# ============================================================
# 弱工具问题 + 可视化 / weak-instrument problem + visualization
# ============================================================
def run_iv(z_strength, seed, N=2000, true=0.5):
    r=np.random.default_rng(seed); ab=r.normal(0,1,N); z=r.normal(0,1,N)
    e=z_strength*z + 0.7*ab + r.normal(0,1,N)               # 工具越弱, z_strength 越小 / weaker z
    w=true*e + 0.8*ab + r.normal(0,1,N)
    s1=sm.OLS(e,sm.add_constant(z)).fit(); eh=s1.fittedvalues
    return sm.OLS(w,sm.add_constant(eh)).fit().params[1], s1.fvalue

strong=[run_iv(0.6,s) for s in range(300)]                  # 强工具 / strong
weak  =[run_iv(0.05,s) for s in range(300)]                 # 弱工具 / weak
strong_est=np.array([x[0] for x in strong]); weak_est=np.array([x[0] for x in weak])
print(f"强工具 / strong (F≈{np.median([x[1] for x in strong]):.0f}): 2SLS 均值 {strong_est.mean():.3f}, 标准差 {strong_est.std():.3f}")
print(f"弱工具 / weak   (F≈{np.median([x[1] for x in weak]):.1f}): 2SLS 均值 {weak_est.mean():.3f}, 标准差 {weak_est.std():.3f}  ← 方差爆炸!")

fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① IV 的 DAG / the IV DAG
def node(a,x,y,t,c="#4C72B0"): a.scatter(x,y,s=1600,c=c,zorder=3,edgecolor="k"); a.text(x,y,t,ha="center",va="center",color="white",fontsize=12,fontweight="bold",zorder=4)
def arr(a,p1,p2,c="#333",st="-|>"): a.annotate("",xy=p2,xytext=p1,arrowprops=dict(arrowstyle=st,lw=2,color=c,shrinkA=20,shrinkB=20))
node(ax[0],0,0,"Z\n工具"); node(ax[0],1,0,"X\n教育"); node(ax[0],2,0,"Y\n工资"); node(ax[0],1.5,1,"U\n能力(未观测)",c="#C44E52")
arr(ax[0],(0,0),(1,0)); arr(ax[0],(1,0),(2,0)); arr(ax[0],(1.5,1),(1,0),c="#C44E52"); arr(ax[0],(1.5,1),(2,0),c="#C44E52")
ax[0].text(0,-0.5,"Z只经X影响Y\n(排他性)",fontsize=8,ha="center"); ax[0].set_xlim(-0.6,2.6); ax[0].set_ylim(-0.8,1.4); ax[0].axis("off"); ax[0].set_title("IV 的因果图 / IV DAG")
# ② 估计对比 / estimate comparison
ax[1].bar(["朴素OLS\nnaive","2SLS","真值\ntruth"],[naive,iv_est,true_return],color=["#C44E52","#55A868","#4C72B0"])
ax[1].axhline(true_return,ls="--",color="#4C72B0")
for i,v in enumerate([naive,iv_est,true_return]): ax[1].text(i,v,f"{v:.2f}",ha="center",va="bottom",fontsize=9)
ax[1].set_title("2SLS 还原真值 / IV recovers truth"); ax[1].set_ylabel("教育回报估计")
# ③ 弱工具方差爆炸 / weak instrument variance
ax[2].hist(np.clip(weak_est,-3,4),bins=40,alpha=0.6,color="#C44E52",label=f"弱工具(std {weak_est.std():.1f})")
ax[2].hist(strong_est,bins=40,alpha=0.7,color="#55A868",label=f"强工具(std {strong_est.std():.2f})")
ax[2].axvline(true_return,color="k",lw=2,label="真值 0.5")
ax[2].set_title("弱工具→估计方差爆炸 / weak instrument = huge variance"); ax[2].set_xlabel("2SLS 估计"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/ci07_viz.png",dpi=80); plt.show()
print("弱工具(F<10)时 2SLS 极不可靠——所以永远先看第一阶段 F 统计量")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **IV 是对付未观测混杂的唯一利器**:当关键混杂(能力、动机)**根本测不到**时,PSM、回归、DiD 都无能为力(前两者要求测到混杂,DiD 要时间维度)。IV 另辟蹊径——借一个"只影响处理、不影响结果"的外部工具,撬出处理里"干净"的变化。本例它精确还原了 0.5,而朴素回归高估到 0.78。**这是 IV 不可替代的价值。**
2. **弱工具是最常见的翻车点**:如果工具对处理的影响太弱(第一阶段 F<10),2SLS 的估计会**方差爆炸**(右图红色分布散得离谱)、且系统性地**偏向有偏的 OLS**。所以**用 IV 第一件事就是看第一阶段 F 统计量**——F>10 是经验底线(严格要求更高)。很多"用了 IV"的研究其实栽在弱工具上。
3. **两个更深的诚实点**:①**排他性假设不可检验、且极易被质疑**——你怎么保证"离大学距离"不通过别的路径(如当地经济)影响工资?一个好 IV 研究,一半功夫花在论证排他性上。②**IV 估的是 LATE 不是 ATE**——它只反映"**被工具影响的那群人(complier)**"的效应。用抽签当工具,估的是"因抽签而参军的人"的效应,不代表所有人。所以 IV 的结论**外推性有限**,报告时要说清楚"这是对谁的效应"。

**English**:
1. **IV is the one weapon for unobserved confounding**: when the key confounder (ability, motivation) is **fundamentally unmeasurable**, PSM, regression, and DiD are helpless (the first two require measured confounders, DiD needs a time dimension). IV takes a different route — borrow an external instrument that "affects treatment but not outcome" to extract the "clean" variation. Here it exactly recovers 0.5, while naive regression overestimates to 0.78. **This is IV's irreplaceable value.**
2. **Weak instruments are the most common failure**: if the instrument's effect on treatment is too weak (first-stage F<10), 2SLS estimates suffer **exploding variance** (the red distribution is wildly spread, right) and systematically **bias toward the biased OLS**. So **the first thing with IV is to check the first-stage F** — F>10 is the empirical floor (stricter thresholds exist). Many "IV studies" actually founder on weak instruments.
3. **Two deeper honest points**: ① **the exclusion restriction is untestable and easily challenged** — how do you guarantee "distance to college" doesn't affect wages through another path (local economy)? A good IV study spends half its effort arguing exclusion. ② **IV estimates the LATE, not the ATE** — it reflects only the effect for **the "compliers" the instrument moves**. Using a lottery as an instrument estimates the effect for "those who served because of the lottery," not everyone. So IV conclusions have **limited external validity**; report clearly "whose effect this is."

> 💼 **实战视角 / Practical angle**
> **中文**:IV 在**互联网因果**里也很有用:①**用"随机推荐/随机曝光"当工具**估"某内容对留存的影响"(推荐是随机的、只通过"是否看到内容"影响留存);②**鼓励设计(encouragement design)**——不能强制用户用新功能,就随机给一部分人发提醒(工具),用 IV 估功能真实效果;③价格弹性(用成本冲击当工具)。落地要点:①**必报第一阶段 F**(证明强工具);②**竭力论证排他性**(画 DAG, 想清楚有没有别的路径);③说清楚估的是 LATE(对 complier 的效应);④过度识别时用 Sargan 检验。面试金句:*"当混杂测不到, 用工具变量:找一个只经处理影响结果的外部变量, 2SLS 两阶段撬出处理的干净变化; 三条件是相关/排他/独立, 弱工具(F<10)会让估计方差爆炸, 且估的是 LATE 不是 ATE。"*
> **English**: IV is also useful in **internet causal inference**: ① use **"random recommendation/exposure" as an instrument** to estimate "a content's effect on retention" (recommendation is random and affects retention only through "saw the content"); ② **encouragement designs** — if you can't force users onto a new feature, randomly send some a reminder (the instrument) and use IV for the feature's true effect; ③ price elasticity (use a cost shock as instrument). Deployment keys: ① **always report the first-stage F** (prove a strong instrument); ② **argue exclusion hard** (draw the DAG, think about other paths); ③ state you estimate the LATE (compliers' effect); ④ use the Sargan test when over-identified. Interview line: *"When confounders are unmeasurable, use an instrument: an external variable affecting the outcome only through the treatment; 2SLS's two stages extract the treatment's clean variation; the three conditions are relevance/exclusion/independence; weak instruments (F<10) explode the variance, and IV estimates the LATE not the ATE."*

---
### 小结 / Summary
- **中文**:IV 对付未观测混杂; 工具三条件=相关(Z→X)、排他(Z只经X影响Y)、独立(Z与混杂无关)。
- **English**: IV handles unobserved confounding; three conditions = relevance (Z→X), exclusion (Z affects Y only via X), independence (Z unrelated to the confounder).
- **中文**:2SLS:①X~Z 得干净的 X̂;②Y~X̂ 得因果效应; 本例还原教育回报 0.5(朴素高估到 0.78)。
- **English**: 2SLS: ① X~Z gives clean X̂; ② Y~X̂ gives the causal effect; here it recovers a return of 0.5 (naive overestimates to 0.78).
- **中文**:弱工具(F<10)方差爆炸; 排他性不可检验; 估 LATE 非 ATE(外推性有限)。
- **English**: Weak instruments (F<10) explode variance; the exclusion restriction is untestable; IV estimates the LATE, not the ATE (limited external validity).
